# 04. Predictor Fearure Engineering: OpenStreetMap

Builds predictor features from OSM (via osmnx) for each of the 1353 hexagons: 

- **Transport access** A: bus,tram stops, U/S Bahn stations
- **Commercial and Leisure activity** B: shops, restaurants, cultural POIs
- **Institutional / workplace density** C: offices, universities, coworking,
- **Green space** D: park/green area proximity

Each feature is a count within a  buffer of each hexagon's centroid. 500m for local amenities and 700-800m for more valuable destinations (rail, culture etc)

Population density is handled separately in the next notebook. 
**Output:** grid with new feature columns, saved to `data/processed/berlin_h3_res8_with_osm_features.geojson`.

**Note:** requires internet access to OSM's Overpass API run locally.


In [1]:
import geopandas as gpd
from pathlib import Path
from shapely import wkt
import pandas as pd
import sys
import osmnx as ox
sys.path.append("../src")
from features_osm import fetch_pois, count_in_buffer

In [2]:
GRID = Path("../data/processed/berlin_h3_res8_with_target.geojson")
BOUNDARY_CSV = Path("../data/external/berlin_boundary.csv")

grid = gpd.read_file(GRID)

boundary_df = pd.read_csv(BOUNDARY_CSV)
berlin_boundary = wkt.loads(boundary_df["location"].iloc[0])

print(f"Grid loaded: {len(grid)} hexagons")
grid.head(2)

Grid loaded: 1353 hexagons


,h3_index,resolution,popular_cafe_count,geometry
0,881f188401fffff,8,0,"POLYGON ((13.16441 52.41029, 13.16222 52.40604..."
1,881f188409fffff,8,0,"POLYGON ((13.16131 52.4175, 13.15911 52.41325,..."


## Transit Features 

Two features - the number of tram and/or bus stops within 500m of each hexagon centroid, and the number of U-bahn/S-bahn stations within 800m. 

Before commiting to any osm tags, this first checks what transit related tags are present and well populated in Berlin's OSM data.


In [3]:
explore_tags = {"public_transport": True, "railway": True, "highway": "bus_stop"}
transit_explore = ox.features_from_polygon(berlin_boundary, explore_tags)

print(f"Total transit-tagged features found: {len(transit_explore)}")
print("\nAvailable columns (tag types present):")
print(transit_explore.columns.tolist())

Total transit-tagged features found: 52764

Available columns (tag types present):
['geometry', 'bench', 'bin', 'bus', 'check_date:shelter', 'highway', 'lit', 'name', 'public_transport', 'ref:BVG', 'shelter', 'tactile_paving', 'website', 'wheelchair', 'note', 'operator', 'railway', 'railway:milestone:catenary_mast', 'railway:milestone:emergency_brake_override', 'railway:position', 'railway:position:exact', 'source', 'railway:local_operated', 'railway:switch', 'railway:switch:electric', 'railway:switch:heated', 'railway:switch:movable_frog', 'railway:switch:resetting', 'railway:turnout_side', 'ref', 'contact:website', 'light_rail', 'network', 'network:short', 'network:wikidata', 'official_name', 'railway:ref', 'railway:station_category', 'station', 'train', 'uic_ref', 'wikidata', 'wikipedia', 'railway:signal:combined', 'railway:signal:combined:deactivated', 'railway:signal:combined:form', 'railway:signal:combined:function', 'railway:signal:combined:height', 'railway:signal:combined:shor

In [4]:
print("public_transport values:")
print(transit_explore["public_transport"].value_counts())

print("\nrailway values:")
print(transit_explore["railway"].value_counts())

public_transport values:
public_transport
stop_position            7986
platform                 7481
station                   452
entrance                  137
destination_display        75
info_board                 73
platform_section_sign      51
platform_access            20
service_center              6
service_point               6
stop_area                   5
waiting_room                5
timetable                   3
ticket_office               3
departures_board            2
no                          2
validator                   2
emergency_exit              1
Name: count, dtype: int64

railway values:
railway
signal                    7067
rail                      4171
switch                    3838
tram_crossing             3249
light_rail                3014
                          ... 
isolated_track_section       1
historic                     1
train_depot                  1
container_terminal           1
street_cabinet               1
Name: count, Length: 63, d

In [5]:
pd.set_option("display.max_rows", None)
print(transit_explore["railway"].value_counts())
pd.reset_option("display.max_rows")

railway
signal                                  7067
rail                                    4171
switch                                  3838
tram_crossing                           3249
light_rail                              3014
tram                                    2279
milestone                               2197
platform                                1303
subway                                  1213
disused                                 1048
razed                                    931
tram_stop                                891
abandoned                                855
stop                                     816
subway_entrance                          707
tram_level_crossing                      701
buffer_stop                              603
level_crossing                           506
platform_edge                            441
railway_crossing                         342
crossing                                 333
station                                  284
po

In [6]:
# check for railway=halt and confirm bus stop counts
print("halt count:", (transit_explore["railway"] == "halt").sum())

print("\nhighway values:")
print(transit_explore["highway"].value_counts())

halt count: 66

highway values:
highway
bus_stop        6262
platform        5791
service           66
footway           42
elevator          26
path              19
crossing          18
residential        9
track              4
steps              4
stop               2
pedestrian         2
cycleway           2
give_way           1
traffic_sign       1
corridor           1
Name: count, dtype: int64


**Tag decision:** the query above shows that the newer `public_transport=*` tagging scheme (`stop_position`, `platform`) tags each physical stop as two separate points and could cause double-counting. The older tagging scheme was used instead, giving one point per stop:

- Bus/tram stops: `highway=bus_stop` (6,262) + `railway=tram_stop` (891)
- Rail stations: `railway=station` (284) + `railway=halt` (66)

`highway=platform` (5,791) is excluded, since it represents a waiting area beside a bus stop instead of a distinct stop.


In [7]:
stop_tags = {"highway": "bus_stop", "railway": "tram_stop"}
stops = fetch_pois(berlin_boundary, stop_tags)
print(f"Bus/tram stops found: {len(stops)}")

grid = count_in_buffer(grid, stops, buffer_m=500, col_name="transit_stops_500m")
grid["transit_stops_500m"].describe()

Bus/tram stops found: 7153


count    1353.000000
mean        6.297857
std         5.960233
min         0.000000
25%         1.000000
50%         5.000000
75%        10.000000
max        50.000000
Name: transit_stops_500m, dtype: float64

In [8]:
rail_tags = {"railway": ["station", "halt"]}
rail = fetch_pois(berlin_boundary, rail_tags)
print(f"Rail stations found: {len(rail)}")

grid = count_in_buffer(grid, rail, buffer_m=800, col_name="rail_stations_800m")
grid["rail_stations_800m"].describe()

Rail stations found: 350


count    1353.000000
mean        0.796009
std         1.418834
min         0.000000
25%         0.000000
50%         0.000000
75%         1.000000
max        12.000000
Name: rail_stations_800m, dtype: float64

## Exploring available commercial/POI tags in Berlin


In [9]:
commercial_explore_tags = {"shop": True, "office": True, "amenity": True}
commercial_explore = ox.features_from_polygon(berlin_boundary, commercial_explore_tags)

print(f"Total commercial features found: {len(commercial_explore)}")

Total commercial features found: 220437


In [10]:
print("Features with a shop tag:", commercial_explore["shop"].notna().sum())
print("Features with an office tag:", commercial_explore["office"].notna().sum())
print("Features with an amenity tag:", commercial_explore["amenity"].notna().sum())

Features with a shop tag: 21423
Features with an office tag: 5850
Features with an amenity tag: 193549


In [11]:
print("shop values:")
print(commercial_explore["shop"].value_counts().head(30))

print("\noffice values:")
print(commercial_explore["office"].value_counts().head(30))

shop values:
shop
hairdresser            2044
clothes                1592
supermarket            1362
bakery                 1286
beauty                 1122
convenience            1112
florist                 680
kiosk                   639
car_repair              502
bicycle                 388
jewelry                 371
car                     368
vacant                  360
optician                355
massage                 342
books                   308
mobile_phone            289
travel_agency           288
furniture               257
shoes                   251
chemist                 248
tailor                  240
interior_decoration     206
beverages               186
funeral_directors       178
deli                    175
cosmetics               173
medical_supply          168
variety_store           163
art                     161
Name: count, dtype: int64

office values:
office
company                    1430
government                  363
association                 3

In [12]:
print("amenity values (top 40):")
print(commercial_explore["amenity"].value_counts().head(40))

amenity values (top 40):
amenity
parking             55530
bench               32948
bicycle_parking     26248
waste_basket        20064
parking_space        7180
vending_machine      5286
restaurant           4751
recycling            2831
fast_food            2830
parking_entrance     2612
cafe                 2592
post_box             2391
kindergarten         2358
charging_station     1716
doctors              1562
shelter              1321
waste_disposal       1257
social_facility      1101
toilets              1100
school               1078
atm                  1055
bar                   975
pub                   893
parcel_locker         839
dentist               824
community_centre      823
place_of_worship      746
pharmacy              669
fountain              606
clock                 597
bicycle_rental        424
taxi                  361
ice_cream             324
bank                  314
hunting_stand         313
driving_school        306
fuel                  292
drink

**Note on `amenity=cafe` (2,592 occurrences ):** cafes are excluded since they are the tafrget variable . This is excluded by construction below, since only `restaurant` and `fast_food` values are selected from `amenity`.

### Junk-checks before building shop and office features

Before finalising `shops_500m` and `offices_500m`, both tag sets are checked for inactive listings

In [13]:
junk_check = ["vacant", "disused", "empty", "closed"]

shop_values = commercial_explore["shop"].value_counts()
print("shop junk values found:")
print(shop_values[shop_values.index.isin(junk_check)])
print("\nTotal shop features:", commercial_explore["shop"].notna().sum())

office_values = commercial_explore["office"].value_counts()
print("\noffice junk values found:")
print(office_values[office_values.index.isin(junk_check)])
print("\nTotal office features:", commercial_explore["office"].notna().sum())

shop junk values found:
shop
vacant    360
Name: count, dtype: int64

Total shop features: 21423

office junk values found:
office
vacant    2
Name: count, dtype: int64

Total office features: 5850


In [14]:
shop_tags = {"shop": True}
shops = fetch_pois(berlin_boundary, shop_tags, exclude={"shop": "vacant"})
print(f"Shops found (excluding vacant): {len(shops)}")

grid = count_in_buffer(grid, shops, buffer_m=500, col_name="shops_500m")
grid["shops_500m"].describe()

Shops found (excluding vacant): 21063


count    1353.000000
mean       18.538803
std        41.969137
min         0.000000
25%         0.000000
50%         3.000000
75%        15.000000
max       432.000000
Name: shops_500m, dtype: float64

In [26]:
office_tags = {"office": True}
offices = fetch_pois(berlin_boundary, office_tags, exclude={"office": "vacant"})
print(f"Offices found (excluding vacant): {len(offices)}")

grid = count_in_buffer(grid, offices, buffer_m=500, col_name="offices_500m")
grid["offices_500m"].describe()

Offices found (excluding vacant): 5848


count    1353.000000
mean        5.181079
std        11.368187
min         0.000000
25%         0.000000
50%         1.000000
75%         4.000000
max       121.000000
Name: offices_500m, dtype: float64

### Food/dining feature

`restaurant` (4,751) and `fast_food` (2,830) were confirmed well-populated in the amenity exploration above. 

In [27]:
food_tags = {"amenity": ["restaurant", "fast_food"]}
food = fetch_pois(berlin_boundary, food_tags)
print(f"Restaurants/fast food found: {len(food)}")

grid = count_in_buffer(grid, food, buffer_m=500, col_name="food_500m")
grid["food_500m"].describe()

Restaurants/fast food found: 7581


count    1353.000000
mean        6.694013
std        16.000493
min         0.000000
25%         0.000000
50%         1.000000
75%         5.000000
max       147.000000
Name: food_500m, dtype: float64

### Universities and coworking spaces

Two additional H2 features added and tags checked


In [28]:
uni_coworking_explore_tags = {"amenity": ["university", "coworking_space"], "office": "coworking"}
uni_coworking_explore = ox.features_from_polygon(berlin_boundary, uni_coworking_explore_tags)

print(f"Total features found: {len(uni_coworking_explore)}")
print("\namenity values:")
print(uni_coworking_explore["amenity"].value_counts(dropna=True))
print("\noffice values:")
print(uni_coworking_explore["office"].value_counts(dropna=True))

Total features found: 266

amenity values:
amenity
university         123
coworking_space     34
cafe                 2
events_venue         1
Name: count, dtype: int64

office values:
office
coworking    111
company        3
yes            1
Name: count, dtype: int64


In [29]:
uni_tags = {"amenity": "university"}
unis = fetch_pois(berlin_boundary, uni_tags)
print(f"Universities found: {len(unis)}")

grid = count_in_buffer(grid, unis, buffer_m=800, col_name="universities_800m")
grid["universities_800m"].describe()

Universities found: 123


count    1353.000000
mean        0.271988
std         1.406322
min         0.000000
25%         0.000000
50%         0.000000
75%         0.000000
max        21.000000
Name: universities_800m, dtype: float64

In [30]:
coworking_tags = {"office": "coworking", "amenity": "coworking_space"}
coworking = fetch_pois(berlin_boundary, coworking_tags)
print(f"Coworking spaces found: {len(coworking)}")

grid = count_in_buffer(grid, coworking, buffer_m=500, col_name="coworking_500m")
grid["coworking_500m"].describe()

Coworking spaces found: 143


count    1353.000000
mean        0.132299
std         0.613771
min         0.000000
25%         0.000000
50%         0.000000
75%         0.000000
max        10.000000
Name: coworking_500m, dtype: float64

## Green space feature 

green spaces function as informal "green third places" alongside cafés and nearby retail/restaurants have shown higher foot-traffic-driven performance near parks.

This feature counts green-space locations (parks, forests, gardens) within 800m of each hexagon's centroid, using the same point-based approach as the other features. This is a simplification: it does not distinguish a single large park from a small one, since each counts as one point at its centroid.

noted as a limitation.

In [31]:
green_explore_tags = {"leisure": "park", "landuse": ["grass", "forest", "recreation_ground"], "natural": "wood"}
green_explore = ox.features_from_polygon(berlin_boundary, green_explore_tags)

print(f"Total green-space features found: {len(green_explore)}")

print("\nleisure values:")
print(green_explore["leisure"].value_counts())
print("\nlanduse values:")
print(green_explore["landuse"].value_counts())
print("\nnatural values:")
print(green_explore["natural"].value_counts())

Total green-space features found: 24338

leisure values:
leisure
park               2673
garden               34
pitch                 3
dog_park              3
nature_reserve        2
fitness_station       2
playground            1
marina                1
beach_resort          1
bathing_place         1
Name: count, dtype: int64

landuse values:
landuse
grass                18770
forest                 970
recreation_ground      129
meadow                   4
religious                2
greenfield               2
fairground               1
cemetery                 1
construction             1
residential              1
Name: count, dtype: int64

natural values:
natural
wood         1880
scrub          69
grassland       4
wetland         3
heath           1
sand            1
shrubbery       1
Name: count, dtype: int64


Final tags used:
- `leisure=park` (2673)
- `landuse=forest` (970) + `natural=wood` (1880) 
  conventions for the same real feature (wooded areas)
- `landuse=recreation_ground` (129)


 `landuse=grass` (18770) is excluded as it includes lawns, street verges etc which is not meaningful green space (noise)

In [32]:
green_tags = {"leisure": "park", "landuse": ["forest", "recreation_ground"], "natural": "wood"}
green_spaces = fetch_pois(berlin_boundary, green_tags)
print(f"Green spaces found: {len(green_spaces)}")

grid = count_in_buffer(grid, green_spaces, buffer_m=800, col_name="green_spaces_800m")
grid["green_spaces_800m"].describe()

Green spaces found: 5626


count    1353.000000
mean       12.301552
std        13.537505
min         0.000000
25%         4.000000
50%         9.000000
75%        17.000000
max       125.000000
Name: green_spaces_800m, dtype: float64

In [33]:
# check raw polygon count vs area, before centroid conversion
raw_check = ox.features_from_polygon(berlin_boundary, {"natural": "wood"})
print(f"Wood polygons: {len(raw_check)}")
print(raw_check.geometry.area.describe())

Wood polygons: 1880
count    1.880000e+03
mean     1.827062e-06
std      1.109610e-05
min      9.510150e-10
25%      1.134392e-07
50%      3.007304e-07
75%      9.890911e-07
max      3.857538e-04
dtype: float64


C:\Windows\Temp\ipykernel_19420\3109523309.py:4: UserWarning: Geometry is in a geographic CRS. Results from 'area' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  print(raw_check.geometry.area.describe())


**Limitation (fragmentation):** a check on the raw wood polygons showed strong fragmentation - areas span about 5 orders of magnitude, with most polygons much smaller than the largest ones. This means large wooded areas (like Grunewald) are mapped in OSM as many small adjacent polygons rather than one shape. Since each polygon is counted separately, hexagons near fragmented forests get inflated green-space counts compared to hexagons near a single, cleanly-mapped park.

## Exploring cultural/leisure POI tags in Berlin 

venues that draw foot traffic through sightseeing, art, or entertainment separate from  green space, food (already built), and shopping (already built). Similar tag check is done as well

In [34]:
culture_explore_tags = {"tourism": ["attraction", "museum", "gallery"], "amenity": ["theatre", "cinema"]}
culture_explore = ox.features_from_polygon(berlin_boundary, culture_explore_tags)

print(f"Total cultural/leisure features found: {len(culture_explore)}")

print("\ntourism values:")
print(culture_explore["tourism"].value_counts())
print("\namenity values:")
print(culture_explore["amenity"].value_counts())

Total cultural/leisure features found: 1120

tourism values:
tourism
gallery       360
museum        247
attraction    229
Name: count, dtype: int64

amenity values:
amenity
theatre             198
cinema               89
fountain             10
place_of_worship      8
townhall              3
planetarium           3
arts_centre           2
events_venue          2
language_school       1
library               1
clock                 1
grave_yard            1
Name: count, dtype: int64


Final tags used:
- `tourism=gallery` (360), `museum` (247), `attraction` (229)
- `amenity=theatre` (198), `cinema` (89)

In [35]:
culture_tags = {"tourism": ["attraction", "museum", "gallery"], "amenity": ["theatre", "cinema"]}
culture = fetch_pois(berlin_boundary, culture_tags)
print(f"Cultural/leisure POIs found: {len(culture)}")

grid = count_in_buffer(grid, culture, buffer_m=700, col_name="culture_700m")
grid["culture_700m"].describe()

Cultural/leisure POIs found: 1120


count    1353.000000
mean        1.921656
std         5.669373
min         0.000000
25%         0.000000
50%         0.000000
75%         1.000000
max        75.000000
Name: culture_700m, dtype: float64

**Plausibility check:** the high maximum (75) likely reflects Berlin's Museum Island and surrounding historic core (Unter den Linden, Brandenburg Gatearea), where museums, galleries, and monuments cluster densely within a small area. Unlike the green-space fragmentation issue, each point here represents a  distinct venue, not a single feature artificially split by mapping.

## Feature summary

| Feature | Group | Buffer |
|---|---|---|
| `transit_stops_500m` | A | 500m |
| `rail_stations_800m` | A | 800m |
| `shops_500m` | B | 500m |
| `food_500m` | B | 500m |
| `culture_700m` | B | 700m |
| `offices_500m` | C | 500m |
| `universities_800m` | C | 800m |
| `coworking_500m` | C | 500m |
| `green_spaces_800m` | D | 800m |

In [36]:
feature_cols = ["transit_stops_500m", "rail_stations_800m", "shops_500m",
                 "offices_500m", "food_500m", "universities_800m",
                 "coworking_500m", "culture_700m", "green_spaces_800m"]

print(grid[feature_cols].describe())

for col in feature_cols:
    assert (grid[col] >= 0).all(), f"{col} has negative counts - something is wrong"
    assert grid[col].sum() > 0, f"{col} is all zero - check the OSM query"

print("\nAll checks passed.")

       transit_stops_500m  rail_stations_800m   shops_500m  offices_500m  \
count         1353.000000         1353.000000  1353.000000   1353.000000   
mean             6.297857            0.796009    18.538803      5.181079   
std              5.960233            1.418834    41.969137     11.368187   
min              0.000000            0.000000     0.000000      0.000000   
25%              1.000000            0.000000     0.000000      0.000000   
50%              5.000000            0.000000     3.000000      1.000000   
75%             10.000000            1.000000    15.000000      4.000000   
max             50.000000           12.000000   432.000000    121.000000   

         food_500m  universities_800m  coworking_500m  culture_700m  \
count  1353.000000        1353.000000     1353.000000   1353.000000   
mean      6.694013           0.271988        0.132299      1.921656   
std      16.000493           1.406322        0.613771      5.669373   
min       0.000000           0.

In [37]:
OUT = Path("../data/processed/berlin_h3_res8_with_osm_features.geojson")
grid.to_file(OUT, driver="GeoJSON")
print(f"Saved {len(grid)} hexagons with {len(feature_cols)} OSM features -> {OUT}")

Saved 1353 hexagons with 9 OSM features -> ..\data\processed\berlin_h3_res8_with_osm_features.geojson
